# Part 09. 프로젝트 루트 확인과 4개 CSV 불러오기

## 35. 프로젝트 루트 설정

In [ ]:

from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":

    project_root = project_root.parent

data_dir = project_root / "data" / "raw"

print("프로젝트 루트:", project_root)

print("데이터 폴더:", data_dir)

print("데이터 폴더 존재:", data_dir.exists())

## 36. pandas와 CSV 불러오기

In [ ]:
import pandas as pd

customers = pd.read_csv(data_dir / "customers.csv")

products = pd.read_csv(data_dir / "products.csv")

orders = pd.read_csv(data_dir / "orders.csv")

order_items = pd.read_csv(data_dir / "order_items.csv")

## 37. 기본 구조와 주요 키 확인

In [ ]:
datasets = {

    "customers": customers,

    "products": products,

    "orders": orders,

    "order_items": order_items,

}

for name, df in datasets.items():

    print(name, df.shape, df.columns.tolist())

In [ ]:

key_checks = {

    "customers.customer_id": customers["customer_id"],

    "products.product_id": products["product_id"],

    "orders.order_id": orders["order_id"],

    "order_items.order_item_id": order_items["order_item_id"],

}

for name, series in key_checks.items():

    print(

        name,

        "결측:", series.isna().sum(),

        "중복:", series.duplicated().sum(),

    )

# Part 10. 컬럼 선택·조건 필터링·정렬

 

## 38. Series와 DataFrame 선택

In [ ]:
city_series = customers["city"]
customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]

print(type(city_series))
print(type(customer_view))
display(customer_view.head())


## 39. 단일 조건 필터링

In [ ]:

customers_over_30 = customers[

    customers["age"] >= 30
]
print(len(customers), len(customers_over_30))
display(customers_over_30.head())

# 40. 복합 조건 필터링

 

30세 이상이면서 서울 거주:

In [ ]:

seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]
display(seoul_over_30.head())


In [ ]:
# 서울 또는 부산 :

seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"])
]
display(
    seoul_or_busan["city"].value_counts()
)

In [ ]:
# 완료 주문이 아닌 주문:

not_completed = orders[
    ~(orders["order_status"] == "completed")
]
display(
    not_completed["order_status"].value_counts(
        dropna=False
    )
)

## 41. 상품 가격 정렬

In [ ]:

expensive_products = (
    products
    .sort_values("price", ascending=False)
    .head(10)
)
display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]
)

# Part 11. line_total 생성과 전체 주문 금액 구분

 

## 42. 작업용 복사본과 파생 컬럼

In [ ]:
order_items_work = order_items.copy()
#데이터 프레임에 새로운 컬럼을 추가할 때는 기존 데이터 프레임을 수정
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)

In [ ]:

display(
    order_items_work[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_total",
        ]
    ].head()
)


In [ ]:
print(order_items_work.head())

In [ ]:
# 43. 수작업 검증

sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]
print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)

In [ ]:
# 44. 전체 주문상세 금액

 

all_order_amount = order_items_work["line_total"].sum()

print("전체 주문상세 금액:", all_order_amount)

In [ ]:
#45. 병합용 주문 컬럼 선택

orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]
].copy()

print(orders_for_merge.shape)
print(orders_for_merge.head())

In [ ]:
#46. 주문상세와 주문 병합

order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )
)

In [ ]:
#47. 병합 검증

print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))
display(
    order_sales["order_match"].value_counts(
        dropna=False
    )
)

In [ ]:
#미매칭 확인:

unmatched_orders = order_sales[
    order_sales["order_match"] != "both"
]
display(unmatched_orders.head())

In [ ]:
#48. 완료 주문 분석셋

display(
    order_sales["order_status"].value_counts(
        dropna=False
    )
)

In [ ]:
completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

In [ ]:
print("완료 주문상세 행:", len(completed_sales))
print(
    "완료 주문 수:",
    completed_sales["order_id"].nunique(),
)
print(
    "완료 주문 고객 수:",
    completed_sales["customer_id"].nunique(),
)
print(
    "완료 주문 매출:",
    completed_sales["line_total"].sum(),
)

In [ ]:
#49. 필요한 상품 정보만 선택

products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]
].copy()

print(products_for_merge.head())

In [ ]:
#50. 완료 주문상세와 상품 병합

completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [ ]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

In [ ]:
#51. 카테고리별 매출

category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

In [ ]:
tmp = order_items.merge(products, on="product_id", how="left", indicator=True)
print(tmp["_merge"].value_counts())

In [ ]:
print("행 수:", len(completed_items))
print("category 결측:", completed_items["category"].isna().sum())

In [ ]:
print(orders["order_status"].unique())
print(orders["order_status"].value_counts())

In [ ]:
#52. 카테고리 합계 검증

category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

In [ ]:
print(orders.columns.tolist())

In [ ]:
print(len(completed_items))

In [ ]:
completed_orders = orders[orders["order_status"] == "실제값"]
print("필터 후 행 수:", len(completed_orders))   # ← 이 한 줄

In [ ]:
print(orders["order_status"].value_counts())

In [ ]:
#50. 완료 주문상세와 상품 병합

completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [ ]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

In [ ]:
#53. 상품별 매출


product_sales = (

    completed_items

    .groupby(

        ["product_id", "product_name", "category"],

        as_index=False,

    )

    .agg(

        total_sales=("line_total", "sum"),

        quantity_sold=("quantity", "sum"),

        order_count=("order_id", "nunique"),

        customer_count=("customer_id", "nunique"),

    )

    .sort_values("total_sales", ascending=False)

)

display(product_sales.head(10))

In [ ]:
#46. 주문상세와 주문 병합

order_sales = (

    order_items_work

    .merge(

        orders_for_merge,

        on="order_id",

        how="left",

        validate="many_to_one",

        indicator="order_match",

    )

)

In [ ]:
#47. 병합 검증

 

print("병합 전 행 수:", len(order_items_work))

print("병합 후 행 수:", len(order_sales))

display(

    order_sales["order_match"].value_counts(

        dropna=False

    )

)

In [ ]:
unmatched_orders = order_sales[

    order_sales["order_match"] != "both"

]

display(unmatched_orders.head())

In [ ]:
#48. 완료 주문 분석셋

 

display(

    order_sales["order_status"].value_counts(

        dropna=False

    )

)

In [ ]:
completed_sales = order_sales[

    order_sales["order_status"] == "completed"

].copy()

In [ ]:
print("완료 주문상세 행:", len(completed_sales))

print(

    "완료 주문 수:",

    completed_sales["order_id"].nunique(),

)

print(

    "완료 주문 고객 수:",

    completed_sales["customer_id"].nunique(),

)

print(

    "완료 주문 매출:",

    completed_sales["line_total"].sum(),

)

In [ ]:
#50. 완료 주문상세와 상품 병합

completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [ ]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

In [ ]:
print(repr(orders["order_status"].unique()))

In [ ]:
# 1. 완료 주문 필터
completed_sales = orders[orders["order_status"] == "배송완료"]
print("완료 주문 수:", len(completed_sales))

# 2. 주문상세 + 상품 + 주문 붙이기
completed_items = (
    order_items
    .merge(products, on="product_id", how="inner")
    .merge(completed_sales, on="order_id", how="inner")
)
print("완료 주문상세 행 수:", len(completed_items))

# 3. 파생 컬럼
completed_items["line_total"] = (
    completed_items["quantity"] * completed_items["unit_price"]
)

# 4. 카테고리별 매출
category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

In [ ]:
#51. 카테고리별 매출

category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
.sort_values("total_sales", ascending=False)
)
display(category_sales)

In [ ]:
category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

In [ ]:
#51. 카테고리별 매출

category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

In [ ]:
#52. 카테고리 합계 검증

category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

In [ ]:
#53. 상품별 매출
product_sales = (
    completed_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        quantity_sold=("quantity", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)
display(product_sales.head(10))

In [ ]:
#54. 주문 날짜 변환과 주문 월 생성

completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),
)

In [ ]:
completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

In [ ]:
print(completed_items["order_date"].dtype)

In [ ]:
completed_items["order_date"] = pd.to_datetime(completed_items["order_date"])

completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

print(completed_items["order_date"].dtype)
print(completed_items[["order_date", "order_month"]].head())

In [ ]:
#54. 주문 날짜 변환과 주문 월 생성

completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),
)

In [ ]:
completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

In [ ]:
#55. 월별 매출

monthly_sales = (
    completed_items
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("order_month")
)
display(monthly_sales)

In [ ]:
#56. 고객별 구매 금액

customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
)

In [ ]:
#57. 고객 속성 연결

#개인정보 최소화를 위해 이름은 제외합니다.
customer_attributes = customers[
    ["customer_id", "gender", "age", "city"]
].copy()

In [ ]:
customer_sales_detail = (
    customer_sales
    .merge(
        customer_attributes,
        on="customer_id",
        how="left",
        validate="one_to_one",
        indicator="customer_match",
    )
    .sort_values("total_sales", ascending=False)
)

In [ ]:
display(
    customer_sales_detail["customer_match"].value_counts(
        dropna=False
    )
)
display(customer_sales_detail.head(10))

In [ ]:
#58. 결과 폴더 생성
output_dir = project_root / "reports" / "chapter04"
output_dir.mkdir(parents=True, exist_ok=True)
print(output_dir)

In [ ]:
#59. 결과 파일 저장
outputs = {
    "category_sales.csv": category_sales,
    "product_sales.csv": product_sales,
    "monthly_sales.csv": monthly_sales,
    "customer_sales.csv": customer_sales_detail,
}
for file_name, df in outputs.items():
    output_path = output_dir / file_name
    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(
        file_name,
        output_path.exists(),
        output_path.stat().st_size,
    )

In [ ]:
#60. 저장 결과 다시 읽기
saved_category_sales = pd.read_csv(
    output_dir / "category_sales.csv"
)
display(saved_category_sales.head())
print(saved_category_sales.shape)

In [ ]:
#61. 병합 점검 함수
def check_merge_result(
    *,
    name: str,
    left_rows: int,
    merged: pd.DataFrame,
    indicator_column: str,
) -> None:
    print(f"[{name}]")
    print("병합 전 행 수:", left_rows)
    print("병합 후 행 수:", len(merged))
    print(
        merged[indicator_column].value_counts(
            dropna=False
        )
    )

In [ ]:
#위에 만든 함수 호출
check_merge_result(
    name="주문상세-주문",
    left_rows=len(order_items_work),
    merged=order_sales,
    indicator_column="order_match",
)


In [ ]:
#62. 집계 합계 검증 함수
def check_total(
    *,
    name: str,
    source_total: float,
    summary_total: float,
) -> None:
    difference = source_total - summary_total
    print(f"[{name}]")
    print("원본 합계:", source_total)
    print("요약 합계:", summary_total)
    print("차이:", difference)

In [ ]:
check_total(
    name="카테고리별 매출",
    source_total=completed_items["line_total"].sum(),
    summary_total=category_sales["total_sales"].sum(),
)

# Part 16. LLM pandas 코드 요청·검증과 최종 점검

## 63. 안전한 LLM 요청 정보

LLM에는 다음 정보를 제공합니다.

* DataFrame 이름
* 한 행의 의미
* 실제 컬럼명
* dtype 요약
* 주요 키와 관계
* 실제 주문 상태값
* 분석 범위
* 원하는 결과 컬럼
* 검증 기준

고객 이름, 이메일, 전화번호, 주소, 원본 주문 행 전체와 API 키는 제공하지 않습니다.

## 64.pandas 코드 요청 프롬프트

나는 온라인 쇼핑몰 데이터를 pandas로 분석하고 있습니다.

분석 목표:
완료 주문 기준 카테고리별 매출을 계산합니다.

DataFrame과 한 행의 의미:
- orders: 주문 한 건
- order_items: 주문에 포함된 상품 한 항목
- products: 상품 한 개

실제 컬럼:
- orders:
  order_id, customer_id, order_date,
  payment_method, order_status
- order_items:
  order_item_id, order_id, product_id,
  quantity, unit_price
- products:
  product_id, product_name, category, price

주요 관계:
- order_items.order_id → orders.order_id
  many_to_one
- order_items.product_id → products.product_id
  many_to_one

분석 범위:
- order_status가 배송완료인 주문만 포함
- line_total = quantity × unit_price
- 주문 수는 order_id의 고유 개수
- 매출은 line_total 합계

원하는 결과:
- category
- total_sales
- order_count
- customer_count
- quantity_sold

검증 요구사항:
1. 각 merge에 validate를 사용해 주세요.
2. indicator로 미매칭을 확인해 주세요.
3. 병합 전후 행 수를 출력해 주세요.
4. 카테고리 합계와 완료 주문 전체 합계를 비교해 주세요.
5. 실제로 존재하지 않는 컬럼을 만들지 마세요.
6. 코드 실행 전 확인할 항목도 설명해 주세요.

## 65. LLM 코드 검증표

| 검증 항목 | 확인 내용 | 결과 | 근거 |
| --- | --- | --- | --- |
| DataFrame | 실제 변수명과 같은가? | ✅ 통과 | orders / order_items / products / customers 일치 |
| 컬럼 | 실제 컬럼만 사용하는가? | ✅ 통과 | 존재하지 않는 컬럼 생성 없음 |
| 상태값 | `completed` 표기가 맞는가? | ✅ 통과 | `COMPLETED = "배송완료"`로 정정, assert로 방어 |
| 계산식 | `quantity × unit_price`인가? | ✅ 통과 | `products.price` 미사용, 수작업 검증 일치 |
| 분석 범위 | 완료 주문만 포함하는가? | ✅ 통과 | `assert len > 0`으로 0행 차단 |
| 주문 수 | `nunique()`를 사용하는가? | ✅ 통과 | `order_count=("order_id", "nunique")` |
| 병합 키 | 실제 관계와 맞는가? | ✅ 통과 | order_id · product_id 모두 both 14,603 |
| validate | `many_to_one`이 적용되었는가? | ✅ 통과 | 주문·상품 병합 모두 적용 |
| indicator | 미매칭을 확인하는가? | ✅ 통과 | left_only 0, right_only 0 |
| 행 수 | 병합 전후를 비교하는가? | ✅ 통과 | 14,603 → 14,603, 증가 없음 |
| 합계 | 원본과 요약 합계를 비교하는가? | ✅ 통과 | 1,481,175,900 = 1,481,175,900, 차이 0 |
| 개인정보 | 원본 고객 정보를 요구하지 않는가? | ✅ 통과 | 집계에서 name 제외, 셀 출력 전체 제거 |


In [ ]:
COMPLETED = "배송완료"

assert COMPLETED in orders["order_status"].unique()

items = order_items.copy()
items["line_total"] = items["quantity"] * items["unit_price"]

before = len(items)
step1 = items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator="order_match",
)
print("주문 병합 전/후:", before, "->", len(step1))
print(step1["order_match"].value_counts(dropna=False))

before = len(step1)
step2 = step1.merge(
    products[["product_id", "product_name", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="product_match",
)
print("상품 병합 전/후:", before, "->", len(step2))
print(step2["product_match"].value_counts(dropna=False))

completed_items = step2[step2["order_status"] == COMPLETED].copy()
assert len(completed_items) > 0
print("완료 주문상세 행:", len(completed_items))
print("완료 주문 수:", completed_items["order_id"].nunique())

completed_items["order_date"] = pd.to_datetime(completed_items["order_date"])
completed_items["order_month"] = (
    completed_items["order_date"].dt.to_period("M").astype("string")
)
print("날짜 변환 실패:", completed_items["order_date"].isna().sum())

### 검증 결과 요약

| 항목 | 값 |
| --- | --- |
| 완료 주문 수 | 4,862 |
| 완료 주문상세 행 수 | 11,764 |
| 완료 주문 총 매출 | 1,481,175,900 |
| 카테고리 수 | 10 |